# Unidad III. Modelos supervisados y análisis exploratorio

**Proyecto:** Análisis y predicción de calidad del aire

Este notebook usa la copia procesada y no modifica `data/raw/`. Se ejecutan regresión lineal, clasificación logística, clustering y PCA con resultados reales.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score, confusion_matrix, silhouette_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
df=pd.read_csv(ROOT/'data/processed/air_quality_etl.csv', parse_dates=['DateTime'])
features=['PT08.S1(CO)','PT08.S2(NMHC)','PT08.S3(NOx)','PT08.S4(NO2)','PT08.S5(O3)','T','RH','AH']
valid=df.dropna(subset=features+['CO(GT)']).sort_values('DateTime').reset_index(drop=True)
valid.shape

(7344, 21)

Se usa una separación temporal 80/20 para evitar mezclar observaciones futuras con el entrenamiento.

In [2]:
split=int(len(valid)*.8); tr=valid.iloc[:split]; te=valid.iloc[split:]
reg=make_pipeline(SimpleImputer(),StandardScaler(),LinearRegression())
reg.fit(tr[features],tr['CO(GT)']); pred=reg.predict(te[features])
regresion={'MAE':mean_absolute_error(te['CO(GT)'],pred),'RMSE':mean_squared_error(te['CO(GT)'],pred)**.5,'R2':r2_score(te['CO(GT)'],pred)}
pd.Series(regresion).round(3)

MAE     0.344
RMSE    0.484
R2      0.862
dtype: float64

La regresión lineal estima la concentración numérica de CO. MAE y RMSE son errores; R² indica qué proporción de variación explica el modelo en el periodo de prueba.

In [3]:
valid['banda_CO']=pd.cut(valid['CO(GT)'],[-np.inf,1,3,np.inf],labels=['Baja','Media','Alta'],right=False)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
clf.fit(tr[features],tr['banda_CO']); clase=clf.predict(te[features])
clasificacion={'accuracy':accuracy_score(te['banda_CO'],clase),'macro_f1':f1_score(te['banda_CO'],clase,average='macro')}
print(pd.Series(clasificacion).round(3)); print(confusion_matrix(te['banda_CO'],clase,labels=['Baja','Media','Alta']))

accuracy    0.865
macro_f1    0.849
dtype: float64
[[241  91   0]
 [ 40 796  23]
 [  0  45 233]]


La clasificación logística asigna una banda de CO. Accuracy es la proporción total de aciertos y macro-F1 promedia el desempeño entre las tres bandas.

In [4]:
Z=StandardScaler().fit_transform(valid[features])
labels=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(Z)
print('Silueta:',round(silhouette_score(Z,labels),3))
pca=PCA(n_components=2,random_state=42).fit(Z)
print('Varianza explicada por PCA:',round(pca.explained_variance_ratio_.sum(),3))

Silueta: 0.265
Varianza explicada por PCA: 0.775


Clustering y PCA son análisis no supervisados: buscan patrones sin utilizar una etiqueta objetivo. Los grupos son exploratorios y no equivalen a categorías sanitarias.